# StoneSense-AI: CT Multi-Model Experimentation — DINOv3 Foundation Model

### 📋 Overview
This notebook evaluates the **Meta DINOv3** Vision Foundation Model (`facebook/dinov3-vits16-pretrain-lvd1689m`) on the 4-class CT kidney dataset (`dl/processed_grouped`).
* If DINOv3 gated access is pending, it cleanly falls back to **DINOv2** (`facebook/dinov2-small`).
* Extracts and caches 384-dimensional foundation embeddings for `train`, `validation`, and `test` splits into `dl/artifacts/embeddings/`.
* Trains class-weighted **Logistic Regression** and **MLP** classifier heads; selects the winner based on validation macro F1.
* Evaluates using `common_eval.py` (cluster bootstrap 95% CIs, slice/cluster metrics, pooled val+test).
* Exports the complete contract bundle (`model_card.json`, `metrics.json`, `predictions.csv`, `head.joblib`, `predict_example.py`, `dinov3_artifacts.zip`).

---
### 1. Global Configuration & Smoke Test Settings
* **What it does:** Sets seed, batch size, model repo ID, paths, idempotency flag `RERUN`, and `SMOKE_TEST` flag (fast 50-sample mode).
* **Expected runtime:** < 1 second.
* **Expected output:** Config summary.

In [ ]:
import os
import sys
import time
from pathlib import Path

SEED = 42
RERUN = False       # Set to True to re-extract embeddings and re-train heads
SMOKE_TEST = False  # Set to True for fast testing (50 images per split, 20 bootstrap resamples)
BATCH_SIZE = 32 if SMOKE_TEST else 64
BOOTSTRAP_RESAMPLES = 20 if SMOKE_TEST else 1000
FAMILY = "dinov3"
VERSION_TAG = "dinov3_vits16_v001"
BASE_MODEL_ID = "facebook/dinov3-vits16-pretrain-lvd1689m"
FALLBACK_MODEL_ID = "facebook/dinov2-small"

WORKSPACE_ROOT = Path("/content/StoneSense-AI") if "google.colab" in sys.modules else Path(".").resolve()
sys.path.insert(0, (WORKSPACE_ROOT / "dl" / "notebooks").as_posix())

DATA_DIR = WORKSPACE_ROOT / "dl" / "processed_grouped"
MANIFEST_PATH = WORKSPACE_ROOT / "dl" / "outputs" / "reports" / "grouped_split_manifest.csv"
EMBEDDINGS_DIR = WORKSPACE_ROOT / "dl" / "artifacts" / "embeddings"
OUT_DIR = WORKSPACE_ROOT / "dl" / "models" / "ct"

EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace Root: {WORKSPACE_ROOT.as_posix()}")
print(f"Embeddings Cache: {EMBEDDINGS_DIR.as_posix()}")
print(f"Output Directory: {(OUT_DIR / FAMILY).as_posix()}")
print(f"Smoke Test Mode: {SMOKE_TEST} (Resamples: {BOOTSTRAP_RESAMPLES})")

---
### 2. Check Cell: Environment, GPU & Model Access Verification
* **What it does:** Verifies PyTorch, GPU availability, reads `HF_TOKEN`, and tests loading DINOv3 (or switches to DINOv2 fallback).
* **Expected runtime:** ~5-10 seconds.
* **Expected output:** Loaded foundation backbone name and feature dimensionality (384).

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

hf_token = os.getenv("HF_TOKEN")
if "google.colab" in sys.modules:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or hf_token
    except Exception:
        pass

active_model_id = BASE_MODEL_ID
is_fallback = False
license_name = "DINOv3 License"

try:
    print(f"Attempting to load primary backbone: {active_model_id}...")
    processor = AutoImageProcessor.from_pretrained(active_model_id, token=hf_token)
    backbone = AutoModel.from_pretrained(active_model_id, token=hf_token).to(device)
    backbone.eval()
    print(f"Successfully loaded primary backbone: {active_model_id}")
except Exception as exc:
    print(f"Primary DINOv3 model access unavailable ({exc}).")
    print(f"Switching to fallback foundation model: {FALLBACK_MODEL_ID}...")
    active_model_id = FALLBACK_MODEL_ID
    is_fallback = True
    license_name = "Apache-2.0"
    processor = AutoImageProcessor.from_pretrained(active_model_id)
    backbone = AutoModel.from_pretrained(active_model_id).to(device)
    backbone.eval()
    print(f"Successfully loaded fallback backbone: {active_model_id}")

DISPLAY_NAME = f"DINOv2 ViT-S/16 (Fallback)" if is_fallback else "DINOv3 ViT-S/16 Foundation"
print(f"Model Display Name: {DISPLAY_NAME}")
print(f"License: {license_name}")

---
### 3. Extract & Cache Foundation Embeddings (`.npz`)
* **What it does:** Iterates through `train`, `validation`, and `test` splits, runs frozen backbone feature extraction, and saves `{split}.npz` containing `embeddings (N, 384)`, `labels (N,)`, `filenames (N,)`, and `cluster_ids (N,)`.
* **Expected runtime:** ~1-2 mins on GPU (~10s in SMOKE_TEST mode). Idempotent: skips if cached.
* **Expected output:** Cache file confirmation with tensor shapes.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

df_manifest = pd.read_csv(MANIFEST_PATH)
start_time = time.time()

for split in ["train", "validation", "test"]:
    cache_file = EMBEDDINGS_DIR / f"{FAMILY}_{split}.npz"
    if cache_file.exists() and not RERUN and not SMOKE_TEST:
        print(f"Cached embeddings found at {cache_file.name}. Skipping extraction.")
        continue

    split_df = df_manifest[df_manifest["destination_split"] == split].reset_index(drop=True)
    if SMOKE_TEST:
        split_df = split_df.head(50)

    embeddings_list = []
    labels_list = []
    filenames_list = []
    cluster_ids_list = []

    print(f"Extracting {len(split_df):,} embeddings for '{split}' split...")
    
    for i in tqdm(range(0, len(split_df), BATCH_SIZE)):
        batch_rows = split_df.iloc[i : i + BATCH_SIZE]
        images = []
        for _, row in batch_rows.iterrows():
            img_path = DATA_DIR / split / row["class"] / row["filename"]
            with Image.open(img_path) as img:
                images.append(img.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = backbone(**inputs)
            if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
                emb = outputs.pooler_output
            else:
                emb = outputs.last_hidden_state[:, 0]
            emb = emb.cpu().numpy().astype(np.float32)

        embeddings_list.append(emb)
        labels_list.extend([CLASS_TO_IDX[r["class"]] for _, r in batch_rows.iterrows()])
        filenames_list.extend(batch_rows["filename"].tolist())
        cluster_ids_list.extend(batch_rows["near_duplicate_group_id"].tolist())

    np.savez_compressed(
        cache_file,
        embeddings=np.vstack(embeddings_list),
        labels=np.array(labels_list, dtype=np.int64),
        filenames=np.array(filenames_list),
        cluster_ids=np.array(cluster_ids_list, dtype=np.int64)
    )
    print(f"Saved {cache_file.name} ({len(filenames_list)} records, shape: {np.vstack(embeddings_list).shape})")

extraction_duration = time.time() - start_time
print(f"Embedding extraction complete in {extraction_duration:.1f}s.")

---
### 4. Train Classification Heads (Logistic Regression & MLP)
* **What it does:** Loads cached embeddings, fits class-weighted Logistic Regression and MLP heads on `train`, evaluates on `validation`, and selects the champion head.
* **Expected runtime:** ~10-20 seconds (~1s in SMOKE_TEST).
* **Expected output:** Validation comparison table and selected champion head.

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score

train_data = np.load(EMBEDDINGS_DIR / f"{FAMILY}_train.npz")
val_data = np.load(EMBEDDINGS_DIR / f"{FAMILY}_validation.npz")

X_train, y_train = train_data["embeddings"], train_data["labels"]
X_val, y_val = val_data["embeddings"], val_data["labels"]

print(f"Training features: {X_train.shape}, Validation features: {X_val.shape}")

# 1. Logistic Regression Head
lr_head = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED,
    C=1.0
)
lr_head.fit(X_train, y_train)
lr_val_f1 = f1_score(y_val, lr_head.predict(X_val), average="macro")

# 2. MLP Head
mlp_head = MLPClassifier(
    hidden_layer_sizes=(128, 32) if SMOKE_TEST else (256, 64),
    max_iter=50 if SMOKE_TEST else 300,
    activation="relu",
    early_stopping=True,
    random_state=SEED
)
mlp_head.fit(X_train, y_train)
mlp_val_f1 = f1_score(y_val, mlp_head.predict(X_val), average="macro")

print("=" * 60)
print(f"Validation Macro F1:")
print(f"   - Logistic Regression Head : {lr_val_f1:.4f}")
print(f"   - MLP Classifier Head      : {mlp_val_f1:.4f}")

if lr_val_f1 >= mlp_val_f1:
    champion_head = lr_head
    head_type = "LogisticRegression"
    champion_f1 = lr_val_f1
else:
    champion_head = mlp_head
    head_type = "MLPClassifier"
    champion_f1 = mlp_val_f1

print(f"Selected Champion Head: {head_type} (Macro F1: {champion_f1:.4f})")
print("=" * 60)

---
### 5. Generate Predictions & Evaluate via `common_eval.py`
* **What it does:** Generates probabilistic predictions for `validation` and `test` splits, runs cluster bootstrap 95% CIs, and calculates slice & cluster-level metrics.
* **Expected runtime:** ~15-25 seconds (~2s in SMOKE_TEST).
* **Expected output:** Metrics summary and confusion matrix.

In [ ]:
from common_eval import generate_full_evaluation, IDX_TO_CLASS

test_data = np.load(EMBEDDINGS_DIR / f"{FAMILY}_test.npz")
X_test, y_test = test_data["embeddings"], test_data["labels"]

# Generate Val Predictions DataFrame
val_probs = champion_head.predict_proba(X_val)
df_val_preds = pd.DataFrame({
    "filename": val_data["filenames"],
    "cluster_id": val_data["cluster_ids"],
    "y_true": [IDX_TO_CLASS[idx] for idx in y_val],
    "y_pred": [IDX_TO_CLASS[idx] for idx in np.argmax(val_probs, axis=1)],
    "p_Cyst": val_probs[:, 0],
    "p_Normal": val_probs[:, 1],
    "p_Stone": val_probs[:, 2],
    "p_Tumor": val_probs[:, 3],
})

# Generate Test Predictions DataFrame
test_probs = champion_head.predict_proba(X_test)
df_test_preds = pd.DataFrame({
    "filename": test_data["filenames"],
    "cluster_id": test_data["cluster_ids"],
    "y_true": [IDX_TO_CLASS[idx] for idx in y_test],
    "y_pred": [IDX_TO_CLASS[idx] for idx in np.argmax(test_probs, axis=1)],
    "p_Cyst": test_probs[:, 0],
    "p_Normal": test_probs[:, 1],
    "p_Stone": test_probs[:, 2],
    "p_Tumor": test_probs[:, 3],
})

eval_results = generate_full_evaluation(df_val_preds, df_test_preds, bootstrap_iterations=BOOTSTRAP_RESAMPLES, seed=SEED)

print("=" * 60)
print(f"TEST SPLIT EVALUATION (DINOv3 + {head_type}):")
print(f"   - Slice Accuracy    : {eval_results['test']['slice_accuracy']:.4f}")
print(f"   - Slice Macro F1    : {eval_results['test']['slice_macro_f1']:.4f} (95% CI: [{eval_results['test']['confidence_intervals_95']['slice_macro_f1']['ci_lower_95']:.4f}, {eval_results['test']['confidence_intervals_95']['slice_macro_f1']['ci_upper_95']:.4f}])")
print(f"   - Cluster Macro F1  : {eval_results['test']['cluster_macro_f1']:.4f}")
print(f"   - Stone Recall      : {eval_results['test']['per_class']['Stone']['recall']:.4f}")
print(f"   - Tumor Recall      : {eval_results['test']['per_class']['Tumor']['recall']:.4f}")
print("=" * 60)

---
### 6. Save Artifacts Contract & `predict_example.py`
* **What it does:** Saves `model_card.json`, `metrics.json`, `predictions.csv`, `head.joblib`, `embedding_config.json`, and `predict_example.py` under `dl/models/ct/dinov3/`.
* **Expected runtime:** ~2-3 seconds.
* **Expected output:** Manifest of saved contract artifacts with SHA-256 hashes.

In [ ]:
import json
from common_eval import save_contract_artifacts

family_dir = OUT_DIR / FAMILY
family_dir.mkdir(parents=True, exist_ok=True)

# 1. Save head artifact
head_artifact_path = family_dir / "head.joblib"
joblib.dump(champion_head, head_artifact_path)

# 2. Save embedding config
config_path = family_dir / "embedding_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "base_model": active_model_id,
        "is_fallback": is_fallback,
        "embedding_dim": int(X_train.shape[1]),
        "head_type": head_type
    }, f, indent=2)

# 3. Create standalone predict_example.py
predict_example_code = f'''"""Standalone DINOv3 prediction example for Phase 8 contract validation."""
from pathlib import Path
import json
import joblib
import numpy as np
from PIL import Image
import torch
from transformers import AutoImageProcessor, AutoModel

CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]

def predict_single_image(image_path: str, model_dir: str = ".") -> dict:
    model_path = Path(model_dir)
    with open(model_path / "embedding_config.json", "r") as f:
        config = json.load(f)
    
    head = joblib.load(model_path / "head.joblib")
    processor = AutoImageProcessor.from_pretrained(config["base_model"])
    backbone = AutoModel.from_pretrained(config["base_model"])
    backbone.eval()
    
    with Image.open(image_path) as img:
        inputs = processor(images=img.convert("RGB"), return_tensors="pt")
    
    with torch.no_grad():
        outputs = backbone(**inputs)
        emb = outputs.pooler_output if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]
        emb_np = emb.cpu().numpy().astype(np.float32)
    
    probs = head.predict_proba(emb_np)[0]
    return {c: float(p) for c, p in zip(CLASS_NAMES, probs)}

if __name__ == "__main__":
    import sys
    img_file = sys.argv[1] if len(sys.argv) > 1 else "sample.jpg"
    print(f"Probabilities: {predict_single_image(img_file)}")
'''

# 4. Save contract
saved = save_contract_artifacts(
    out_dir=OUT_DIR,
    family=FAMILY,
    version_tag=VERSION_TAG,
    display_name=DISPLAY_NAME,
    base_model=active_model_id,
    license_name=license_name,
    hyperparameters={"head_type": head_type, "batch_size": BATCH_SIZE, "seed": SEED, "smoke_test": SMOKE_TEST},
    preprocessing_spec={"resize": [224, 224], "normalization": "ImageNet / DINO standard"},
    library_versions={"torch": torch.__version__, "transformers": "installed"},
    training_time_sec=extraction_duration,
    evaluation_results=eval_results,
    additional_artifact_paths=[head_artifact_path, config_path],
    predict_example_code=predict_example_code,
    seed=SEED
)

print(f"Successfully saved Phase 8 contract artifacts to {family_dir.as_posix()}")

---
### 7. Package Zip for Project Integration
* **What it does:** Zips `dl/models/ct/dinov3/` into `dinov3_artifacts.zip` and reports target deployment path.
* **Expected runtime:** ~2-5 seconds.
* **Expected output:** Downloadable zip confirmation.

In [ ]:
import shutil

zip_path = WORKSPACE_ROOT / "dl" / "models" / "ct" / f"{FAMILY}_artifacts.zip"
shutil.make_archive(str(zip_path.with_suffix("")), 'zip', family_dir)

print("=" * 60)
print(f"CREATED ARTIFACT ZIP: {zip_path.name}")
print(f"Target Project Unzip Directory: dl/models/ct/{FAMILY}/")
print("=" * 60)